# PID Tuning

Systematic parameter sweep to find optimal PID gains.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, numpy as np, matplotlib.pyplot as plt, json
sys.path.append('..')
from src.control.pid import PIDController

print('Ready')

In [ ]:
def simulate(Kp, Ki, Kd, target=1.0, steps=100):
    pid = PIDController(Kp=Kp, Ki=Ki, Kd=Kd, output_limits=(-30, 30))
    error = target
    errors, outputs = [], []
    for _ in range(steps):
        output = pid.update(error)
        errors.append(error)
        outputs.append(output)
        error = target - output * 0.1
        if error < 0.01:
            error = 0
    return errors

def settling_time(errors, threshold=0.05):
    for i in range(len(errors)):
        if all(abs(e) < threshold for e in errors[i:]):
            return i
    return len(errors)

In [ ]:
# Sweep
param_grid = {'Kp': [0.5, 1.0, 2.0, 3.0, 4.0],
              'Ki': [0.0, 0.05, 0.1],
              'Kd': [0.0, 0.5, 1.0]}

results = []
for Kp in param_grid['Kp']:
    for Ki in param_grid['Ki']:
        for Kd in param_grid['Kd']:
            errors = simulate(Kp, Ki, Kd)
            settle = settling_time(errors)
            overshoot = max(0, max(errors))
            steady = abs(np.mean(errors[-10:])) if len(errors) >= 10 else abs(errors[-1])
            results.append({'Kp': Kp, 'Ki': Ki, 'Kd': Kd,
                           'settling': settle, 'overshoot': overshoot,
                           'steady_error': steady})

best = min(results, key=lambda r: r['settling'])
print(f'Best: Kp={best["Kp"]}, Ki={best["Ki"]}, Kd={best["Kd"]}')
print(f'  Settling: {best["settling"]} steps')
print(f'  Overshoot: {best["overshoot"]:.4f}')
print(f'  Steady error: {best["steady_error"]:.4f}')

In [ ]:
# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for idx, (Kp_val, ax) in enumerate(zip([0.5, 1.0, 2.0, 4.0], axes.flat)):
    for Kd_val in [0.0, 0.5, 1.0]:
        errors = simulate(Kp_val, 0.05, Kd_val)
        ax.plot(errors, label=f'Kd={Kd_val}')
    ax.set_title(f'Kp={Kp_val}, Ki=0.05')
    ax.set_xlabel('Step')
    ax.set_ylabel('Error')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Save results
os.makedirs('../reports/logs', exist_ok=True)
with open('../reports/logs/pid_tuning.json', 'w') as f:
    json.dump({'sweep_results': results, 'best_params': best}, f, indent=2)
print('Saved to reports/logs/pid_tuning.json')